# Can we predict whether an account is in arrears next month?

Target: `arrears_next_month`. Each row is an account-month; the model would be used on *future* billing months, for accounts it may not have seen. Methodology follows `checklist.md`.

**Feature decisions (checklist item 7).** `columns.csv` says `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time (and is nearly a proxy for the outcome). It is dropped, as is the identifier `meter_ref` (used only for grouping). Remaining features: `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score

df = pd.read_csv('data.csv', parse_dates=['billing_month'])
print(df.shape, 'prevalence', df.arrears_next_month.mean().round(4))
print('accounts', df.meter_ref.nunique(), '| months', df.billing_month.min().date(), '->', df.billing_month.max().date())
print('rows with final notice:', df.final_notice_sent.mean().round(3), '| arrears rate if notice=1:', df[df.final_notice_sent==1].arrears_next_month.mean().round(3), 'else', df[df.final_notice_sent==0].arrears_next_month.mean().round(3))

## 1. Test set, set aside before any modelling
Accounts appear in many rows and the model is meant for later months, so the test set respects both: 20% of accounts (random, fixed seed) are held out entirely, and only their rows in the **latest 6 months** form the test set. Training rows are all rows of the other accounts *before* that 6-month window. No test account appears in training, and no training row is later than a test row.

In [ ]:
FEATURES = ['tariff', 'occupants', 'kwh', 'bill_eur', 'late_payments_12m']
TARGET = 'arrears_next_month'
rng = np.random.default_rng(0)
accts = np.sort(df.meter_ref.unique())
test_accts = set(rng.choice(accts, size=int(0.2 * len(accts)), replace=False))
months = np.sort(df.billing_month.unique())
test_start = months[-6]
is_test = df.meter_ref.isin(test_accts) & (df.billing_month >= test_start)
is_train = ~df.meter_ref.isin(test_accts) & (df.billing_month < test_start)
test, train = df[is_test].reset_index(drop=True), df[is_train].reset_index(drop=True)
assert not set(train.meter_ref) & set(test.meter_ref) and train.billing_month.max() < test.billing_month.min()
print('train', train.shape, 'prev', train[TARGET].mean().round(4), '| test', test.shape, 'prev', test[TARGET].mean().round(4))

## 2. Cross-validation inside the training rows only
Same structure as the test split: expanding-window forward chaining over months. Each fold validates on a block of 3 later months for a random third of the training accounts, and trains on all earlier months of the *other* accounts (accounts never straddle train/validation within a fold). Five folds.

In [ ]:
def cv_splits(d, n_folds=5, block=3, frac=1/3, seed=0):
    r = np.random.default_rng(seed)
    m = np.sort(d.billing_month.unique())
    a = np.sort(d.meter_ref.unique())
    for k in range(n_folds):
        v0 = len(m) - block * (n_folds - k)
        vm = m[v0:v0 + block]
        val_accts = set(r.choice(a, size=int(frac * len(a)), replace=False))
        in_val = d.meter_ref.isin(val_accts)
        val = d.billing_month.isin(vm) & in_val
        tr = (d.billing_month < vm[0]) & ~in_val
        yield np.where(tr)[0], np.where(val)[0]

for i, (a, b) in enumerate(cv_splits(train)):
    print(i, 'train', len(a), 'val', len(b), 'val prev', train[TARGET].iloc[b].mean().round(3))

## 3. Candidate models (named before any scoring)
1. **baseline**: predicts the training prevalence for every row (`DummyClassifier(prior)`).
2. **logreg**: logistic regression, one-hot tariff, standardised numerics; C in {0.1, 1, 10}.
3. **hgb**: histogram gradient boosting; (max_depth, learning_rate) in {(2, 0.05), (3, 0.1)}, 150 iterations.

Preprocessing (scaling, encoding) lives inside each pipeline, so it is fitted on each CV training fold only (checklist 3).
Selection metric: mean CV average precision (PR-AUC; the positive class is ~12%). The decision threshold is picked on pooled out-of-fold predictions of the winner to maximise F1. This rule is fixed here, before scoring.

In [ ]:
num = ['occupants', 'kwh', 'bill_eur', 'late_payments_12m']
def prep(scale):
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['tariff']),
                              ('num', StandardScaler() if scale else 'passthrough', num)])

candidates = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))])}
for C in [0.1, 1, 10]:
    candidates['logreg_C%s' % C] = Pipeline([('p', prep(True)), ('m', LogisticRegression(C=C, max_iter=1000))])
for d, lr in [(2, 0.05), (3, 0.1)]:
    candidates['hgb_d%d_lr%s' % (d, lr)] = Pipeline([('p', prep(False)),
        ('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=lr, max_iter=150, random_state=0))])
print(list(candidates))

In [ ]:
X, y = train[FEATURES], train[TARGET].values
rows, oof = [], {}
for name, est in candidates.items():
    p = np.full(len(train), np.nan); aps, aucs = [], []
    for tr, va in cv_splits(train):
        m = clone(est).fit(X.iloc[tr], y[tr]); pv = m.predict_proba(X.iloc[va])[:, 1]
        p[va] = pv; aps.append(average_precision_score(y[va], pv)); aucs.append(roc_auc_score(y[va], pv))
    oof[name] = p
    rows.append((name, np.mean(aps), np.std(aps), np.mean(aucs)))
cv = pd.DataFrame(rows, columns=['model', 'cv_AP', 'cv_AP_sd', 'cv_AUC']).sort_values('cv_AP', ascending=False)
cv.round(4)

## 4. Choice, written down before the test set is touched

In [ ]:
best = cv.iloc[0].model
mask = ~np.isnan(oof[best]); ths = np.linspace(0.02, 0.9, 89)
f1s = [f1_score(y[mask], oof[best][mask] >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
choice = dict(model=best, threshold=thr, cv_AP=float(cv.iloc[0].cv_AP), cv_F1_at_threshold=float(max(f1s)))
json.dump(choice, open('chosen_model.json', 'w'))
print('CHOSEN (selected on CV only):', choice)

## 5. Final fit and single test evaluation
Refit the chosen pipeline on all training rows and score the test set once. Nothing is changed after this cell.

In [ ]:
final = clone(candidates[choice['model']]).fit(X, y)
Xt, yt = test[FEATURES], test[TARGET].values
pt = final.predict_proba(Xt)[:, 1]; pred = pt >= choice['threshold']
base = clone(candidates['baseline']).fit(X, y).predict_proba(Xt)[:, 1]
res = pd.DataFrame({'chosen': [average_precision_score(yt, pt), roc_auc_score(yt, pt), precision_score(yt, pred), recall_score(yt, pred), f1_score(yt, pred)],
                    'baseline (prevalence)': [average_precision_score(yt, base), roc_auc_score(yt, base), np.nan, np.nan, np.nan]},
                   index=['PR-AUC', 'ROC-AUC', 'precision@thr', 'recall@thr', 'F1@thr'])
print('test rows', len(yt), 'positive rate', yt.mean().round(4)); res.round(4)

In [ ]:
# Uncertainty: bootstrap over test accounts (rows within an account are correlated)
g = test.meter_ref.values; ua = np.unique(g); idx = {a: np.where(g == a)[0] for a in ua}
br = np.random.default_rng(1); aps = []; aucs = []
for _ in range(500):
    s = np.concatenate([idx[a] for a in br.choice(ua, len(ua))])
    aps.append(average_precision_score(yt[s], pt[s])); aucs.append(roc_auc_score(yt[s], pt[s]))
print('PR-AUC 95%% CI: %.3f-%.3f | ROC-AUC 95%% CI: %.3f-%.3f' % (*np.percentile(aps, [2.5, 97.5]), *np.percentile(aucs, [2.5, 97.5])))

## 6. Conclusion
The headline is the test PR-AUC / ROC-AUC above, compared with the prevalence baseline. Caveats: the test window is a single 6-month period with a limited number of held-out accounts, so intervals are wide; `final_notice_sent` was deliberately excluded because it is not available at prediction time, which makes this a harder but honest problem than a model using it would suggest.